In [1]:
!pip install transformers pillow tqdm requests faiss-cpu

import os
import json
import torch
import random
import numpy as np
from PIL import Image
from io import BytesIO
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from transformers import CLIPProcessor, CLIPModel
from torch.optim import AdamW
import torch.nn.functional as F
import requests
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import AutoTokenizer as NLLBTokenizer
from transformers import AutoModelForSeq2SeqLM


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 69.2 MB/s eta 0:00:00:00:0100:01
Device: cuda


In [2]:
import numpy as np

# Load embeddings and URLs
image_embeddings = np.load("/kaggle/input/datasets/nadinemohsen804/all-embeddings/image_embeddings (2).npy")   # shape: (N, dim)
text_embeddings = np.load("/kaggle/input/datasets/nadinemohsen804/all-embeddings/text_embeddings (2).npy")     # if needed
all_image_urls = []
with open("/kaggle/input/datasets/nadinemohsen804/all-embeddings/image_urls (2).jsonl", "r", encoding="utf-8") as f:
    for line in f:
        url = line.strip().strip('"')   # removes "quotes"
        all_image_urls.append(url)

# Verify shapes
print("Image embeddings:", image_embeddings.shape)
print("Text embeddings:", text_embeddings.shape if 'text_embeddings' in locals() else "N/A")
print("URLs count:", len(all_image_urls))


Image embeddings: (28026, 512)
Text embeddings: (28026, 512)
URLs count: 28026


In [3]:
!pip install torch torchvision transformers faiss-cpu pillow requests
import faiss
embedding_dim = image_embeddings.shape[1]

index = faiss.IndexFlatIP(embedding_dim)  # cosine similarity (normalized vectors)
index.add(image_embeddings)

print(f"FAISS index built with {index.ntotal} vectors.")


FAISS index built with 28026 vectors.


In [4]:
import torch
from transformers import CLIPProcessor, CLIPModel
import time
def search_text(query_text, top_k=5,measure_time=True):
    start = time.perf_counter() if measure_time else None
    inputs = clip_processor(text=[query_text], return_tensors="pt").to(device)

    with torch.no_grad():
        out = clip_model.get_text_features(
            inputs["input_ids"],
            inputs["attention_mask"]
        )

        # New fix: extract the actual embedding tensor
        if hasattr(out, "pooler_output"):
            q = out.pooler_output
        else:
            q = out.last_hidden_state.mean(dim=1)

        q = q / q.norm(p=2, dim=-1, keepdim=True)

    q = q.cpu().numpy().astype("float32")

    D, I = index.search(q, top_k)

    results = []
    for idx in I[0]:
        if 0 <= idx < len(all_image_urls):
            results.append(all_image_urls[idx])

    if measure_time:
        elapsed = time.perf_counter() - start
        return results, elapsed
        
    return results


def search_image_to_image(query_image_pil, top_k=5,measure_time=True):
    start = time.perf_counter() if measure_time else None
    clip_model.eval()

    inputs = clip_processor(
        images=[query_image_pil],
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        out = clip_model.get_image_features(inputs["pixel_values"])

        # If the model returns an object instead of a tensor
        if not isinstance(out, torch.Tensor):
            if hasattr(out, "pooler_output"):
                img_emb = out.pooler_output
            else:
                img_emb = out.last_hidden_state.mean(dim=1)
        else:
            img_emb = out

        img_emb = img_emb / img_emb.norm(dim=-1, keepdim=True)
        img_emb = img_emb.cpu().numpy().astype("float32")

    D, I = index.search(img_emb, top_k)
    results = [all_image_urls[i] for i in I[0]]

    if measure_time:
        elapsed = time.perf_counter() - start
        return results, elapsed

    return results

In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

qwen_name = "Qwen/Qwen2-1.5B-Instruct"

tokenizer_qwen = AutoTokenizer.from_pretrained(qwen_name)
model_qwen = AutoModelForCausalLM.from_pretrained(qwen_name).to(device)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [7]:
clip_processor = CLIPProcessor.from_pretrained("/kaggle/input/models/nadinemohsen804/trial2-/pytorch/default/1")
clip_model = CLIPModel.from_pretrained("/kaggle/input/models/nadinemohsen804/trial2-/pytorch/default/1").to(device)

clip_model.eval()




Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1e-05,

In [58]:
import re

def contains_arabic(text):
    return any('\u0600' <= c <= '\u06FF' for c in text)

def split_text(text):
    arabic_part = re.findall(r'[\u0600-\u06FF\s]+', text)
    english_part = re.sub(r'[\u0600-\u06FF\s]+', '', text)

    return " ".join(arabic_part).strip(), english_part.strip()

def clean_query(text):
    return " ".join(list(set(text.split())))

def is_english(text):
    return all(ord(c) < 128 for c in text)


def process_query(text):
    arabic, english = split_text(text)

    if arabic:
        # Step 1: apply mapping (VERY IMPORTANT)
        mapped_arabic = normalize_fashion_terms(arabic)

        # Step 2: generate short English query from FULL text
        short_query = generate_search_query(mapped_arabic)

        # 🚨 Step 3: FORCE important keywords to stay
        for en_word in FASHION_MAP.values():
            if en_word in mapped_arabic and en_word not in short_query:
                short_query = en_word + " " + short_query

        final_query = f"{english} {short_query}".strip()

    else:
        final_query = generate_search_query(english)

    return clean_query(final_query)

In [67]:
FASHION_MAP = {
    # 👗 Headwear
    "طرحة": "hijab",
    "حجاب": "hijab",
    "ايشارب": "scarf",
    "سكارف": "scarf",
    "توربان": "turban",
     "طرحه": "hijab",

    # 👚 Tops
    "تيشيرت": "t-shirt",
    "تيشرت": "t-shirt",
    "تشيرت": "t-shirt",
    "بلوزة": "blouse",
    "قميص": "shirt",
    "هودي": "hoodie",
    "سويت شيرت": "sweatshirt",
    "بلوفر": "sweater",
    "كارديجان": "cardigan",
    "جاكيت": "jacket",
    "چاكيت": "jacket",

    # 👗 Dresses
    "فستان": "dress",
    "فستان سواريه": "evening dress",
    "سواريه": "evening dress",
    "فستان سهره": "evening dress",
    "فستان خطوبة": "engagement dress",
    "فستان فرح": "wedding dress",

    # 👖 Bottoms
    "بنطلون": "pants",
    "بنطلون جينز": "jeans",
    "جينز": "jeans",
    "سكيني": "skinny jeans",
    "واسع": "wide pants",
    "جيب": "skirt",
    "جيبه": "skirt",
    "جيبة": "skirt",
    "شورت": "shorts",

    # 👟 Shoes
    "كوتشي": "sneakers",
    "جزمة": "shoes",
    "بوت": "boots",
    "صندل": "sandals",
    "شبشب": "slippers",
    "كعب": "heels",

    # 🎒 Accessories
    "شنطة": "bag",
    "شنطه": "bag",
    "حقيبة": "bag",
    "نضارة": "sunglasses",
    "نظارة": "sunglasses",
    "سلسلة": "necklace",
    "خاتم": "ring",

    # 🎨 Colors (VERY IMPORTANT for search)
    "ابيض": "white",
    "بيضه": "white",
    "اسود": "black",
    "سودا": "black",
    "احمر": "red",
    "ازرق": "blue",
    "كحلي": "navy",
    "اخضر": "green",
    "اصفر": "yellow",
    "بيج": "beige",
    "بني": "brown",
    "رمادي": "gray",
    "موف": "purple",
    "روز": "pink",
    "لبني": "light blue",

    # 📏 Styles / fits
    "واسع": "oversized",
    "اوفر سايز": "oversized",
    "ضيق": "tight",
    "كاجوال": "casual",
    "شيك": "elegant",
    "سبور": "sport",
    "كلاسيك": "classic",
    "سادة": "plain",
    "مقلم": "striped",
    "مشجر": "floral",

    # 🧵 Materials
    "قطن": "cotton",
    "جينز": "denim",
    "جلد": "leather",
    "صوف": "wool",

    # 👰 Occasions
    "خروج": "casual outfit",
    "سهره": "evening",
    "جامعة": "casual",
    "شغل": "formal",
}

In [49]:
@torch.no_grad()

def normalize_fashion_terms(text):
    for ar, en in FASHION_MAP.items():
        if ar in text:
            text = text.replace(ar, en)
    return text
def normalize_egyptian_to_formal(text):
    prompt = f"""
Convert Egyptian Arabic fashion text into formal Arabic.

Text: {text}
Formal Arabic:
"""

    inputs = tokenizer_qwen(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model_qwen.generate(
            **inputs,
            max_new_tokens=30,
            do_sample=False,
            eos_token_id=tokenizer_qwen.eos_token_id
        )

    result = tokenizer_qwen.decode(outputs[0], skip_special_tokens=True)

    # extract only answer
    if "Formal Arabic:" in result:
        result = result.split("Formal Arabic:")[-1]

    return result.strip()

def generate_search_query(text):
    prompt = f"""
Convert this into a short English fashion search query (max 5 words only).

Text: {text}
Query:
"""

    inputs = tokenizer_qwen(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model_qwen.generate(
            **inputs,
            max_new_tokens=15,
            do_sample=False,
            eos_token_id=tokenizer_qwen.eos_token_id
        )

    result = tokenizer_qwen.decode(outputs[0], skip_special_tokens=True)

    if "Query:" in result:
        result = result.split("Query:")[-1]

    return result.strip()

In [9]:
query =  "printed scarf"
results = search_text(query, top_k=5)

for r in results:
    print(r)


['https://cdn.shopify.com/s/files/1/0854/0769/5145/files/DD5C7001-0A4A-4011-803D-57FC723E51FA.jpg', 'https://cdn.shopify.com/s/files/1/0852/8484/7908/files/1_60239f7d-fa35-4a4a-af6a-7748b5d99426.jpg', 'https://cdn.shopify.com/s/files/1/0852/8484/7908/files/1_c4eaaed7-abea-457a-994f-9601f6c81073.jpg', 'https://cdn.shopify.com/s/files/1/0852/8484/7908/files/10411001000503.jpg', 'https://cdn.shopify.com/s/files/1/0852/8484/7908/files/1_39b4e2a7-9671-4a7b-80ef-0fdda2cdc295.jpg']
0.5483344229999716


In [10]:
query_img = Image.open("/kaggle/input/datasets/nadinemohsen804/image5/6303DB83-65CF-478E-8956-E05758945F34.webp").convert("RGB")
result_urls = search_image_to_image(query_img, top_k=5)

for url in result_urls:
    print(url)


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/datasets/nadinemohsen804/image5/6303DB83-65CF-478E-8956-E05758945F34.webp'

In [68]:
import torch
import time
import gradio as gr
import numpy as np
from PIL import Image
from transformers import CLIPProcessor, CLIPModel


# ------------------------------
#   WRAPPER FUNCTIONS FOR GRADIO
# ------------------------------


import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# -----------------------
# Load NLLB safely
# -----------------------
nllb_name = "facebook/nllb-200-distilled-600M"

nllb_tokenizer = AutoTokenizer.from_pretrained(
    nllb_name,
    use_fast=False   # IMPORTANT FIX
)

nllb_model = AutoModelForSeq2SeqLM.from_pretrained(
    nllb_name,
    torch_dtype=torch.float32,
    device_map="cpu"   # safe for Kaggle / Colab / 14GB GPU
)

# target language
TGT_LANG = "eng_Latn"


# -----------------------
# Translation function
# -----------------------
@torch.no_grad()
def translate_ar_to_en(text):
    inputs = nllb_tokenizer(text, return_tensors="pt")

    inputs = {k: v.to("cpu") for k, v in inputs.items()}

    outputs = nllb_model.generate(
        **inputs,
        forced_bos_token_id=nllb_tokenizer.convert_tokens_to_ids("eng_Latn"),
        max_length=40
    )

    return nllb_tokenizer.decode(outputs[0], skip_special_tokens=True)
def search_wrapper_text(text):
    if not text:
        return [], "Please enter a query."

    processed_text = process_query(text)

    print("Final query:", processed_text)

    results, elapsed = search_text(processed_text, top_k=12, measure_time=True)

    return results, f"⏱ Retrieval time: {elapsed:.4f} seconds"


def search_wrapper_image(img):
    if img is None:
        return [], "No image uploaded."

    results, elapsed = search_image_to_image(img, top_k=12, measure_time=True)
    return results, f"⏱ Retrieval time: {elapsed:.4f} seconds"


# ------------------------------
#   CUSTOM CSS
# ------------------------------

custom_css = """
body { background-color: #FFD700; font-family: 'Courier New', Courier, monospace; }
.container { border: 4px solid black; background-color: white; padding: 20px; box-shadow: 10px 10px 0px black; }
.bold-title { text-align: center; color: black; font-size: 3rem; font-weight: 900; margin-bottom: 0; }
.stay-local { text-align: center; color: #E91E63; font-size: 3.5rem; font-weight: 900; margin-top: -10px; }
.match-button { background-color: #FF8C00 !important; color: black !important; font-weight: bold !important; border: 3px solid black !important; box-shadow: 4px 4px 0px black; }
.match-button:hover { transform: translate(-2px, -2px); box-shadow: 6px 6px 0px black; }
.input-box { border: 3px solid black !important; background-color: #FFE4E1 !important; }
"""


# ------------------------------
#   FULL GRADIO UI
# ------------------------------

with gr.Blocks(css=custom_css) as demo:
    with gr.Column(elem_classes="container"):
        gr.Markdown("### LOCAL_SEARCH_AI.EXE")
        gr.Markdown("# FIND THE LOOK,", elem_classes="bold-title")
        gr.Markdown("# STAY LOCAL.", elem_classes="stay-local")
        
        with gr.Row():
            # Left: Image
            with gr.Column():
                img_input = gr.Image(label="UPLOAD IMAGE", type="pil", elem_classes="input-box")
                img_btn = gr.Button("MATCH ME ⚡", elem_classes="match-button")

            gr.Markdown("<div style='text-align:center; font-weight:bold; margin-top:100px;'>OR</div>")

            # Right: Text
            with gr.Column():
                text_input = gr.Textbox(
                    label="DESCRIBE THE VIBE", 
                    placeholder="e.g. 'Oversized linen shirt'...",
                    lines=5,
                    elem_classes="input-box"
                )
                text_btn = gr.Button("MATCH ME ⚡", elem_classes="match-button")

        # Results gallery
        results_gallery = gr.Gallery(
            show_label=False,
            columns=4, rows=3, height=600,
            object_fit="contain"
        )

        # Retrieval time UI
        retrieval_time = gr.Textbox(
            label="Retrieval Time",
            interactive=False
        )

        # Button connections
        img_btn.click(
            fn=search_wrapper_image,
            inputs=img_input,
            outputs=[results_gallery, retrieval_time]
        )

        text_btn.click(
            fn=search_wrapper_text,
            inputs=text_input,
            outputs=[results_gallery, retrieval_time]
        )

# Launch
demo.launch()

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
/tmp/ipykernel_57/4211349584.py:93: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() inste

* Running on local URL:  http://127.0.0.1:7868
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

* Running on public URL: https://0329552500bd59ef7b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Final query: white "White This most search relevant is and concise the for sale" dress
Final query: online" hijabs white This correct "Find sale answer. is the for
Final query: This "Floral most floral relevant is dresses and concise the for sale"
